In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import pystac_client
import planetary_computer as pc
import xarray as xr

def random_date(start_date, end_date):
    """Generate a random date between start_date and end_date."""
    delta = end_date - start_date
    random_days = np.random.randint(0, delta.days + 1)
    return start_date + timedelta(days=random_days)


catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

search = catalog.search(
    collections=["deltares-water-availability"],
    query={"deltares:reservoir": {"eq": "ERA5"}}
)
items = search.item_collection()
if not items:
    raise ValueError("No ERA5 item found.")

item = items[0]
asset = item.assets["data"]
signed_href = pc.sign(asset.href) + '#mode=bytes'

ds = xr.open_dataset(signed_href, engine='netcdf4')
ds = ds.sel(time=slice("2010-01-01", "2015-12-31"))

grand_ids = ds['GrandID'].values
variables = ['P', 'ETa', 'PET', 'Melt', 'Snow', 'Temp', 'P_res', 'S_res', 'Ea_res', 'Qin_res', 'FracFull', 'Qout_res']


np.random.seed(42)  
num_samples = 200
samples = []
start_date = datetime(2010, 1, 1)
end_date = datetime(2015, 12, 31)
times = pd.to_datetime(ds['time'].values)

for _ in tqdm(range(num_samples), desc="Collecting samples"):
    grand_id = np.random.choice(grand_ids)
    date = random_date(start_date, end_date)
    nearest_time_idx = np.argmin(np.abs(times - pd.to_datetime(date)))
    nearest_time = times[nearest_time_idx]
    
    data_dict = {
        'GrandID': grand_id,
        'Latitude': float(ds['latitude'].sel(GrandID=grand_id).values),
        'Longitude': float(ds['longitude'].sel(GrandID=grand_id).values),
        'Sample Date': nearest_time,
    }
    
    for var in variables:
        var_data = ds[var].isel(time=nearest_time_idx).sel(GrandID=grand_id).mean(dim='ksathorfrac').values
        data_dict[var] = float(var_data) if not np.isnan(var_data) else np.nan
    
    samples.append(data_dict)

df = pd.DataFrame(samples)

df = df.dropna()
print(f"Collected {len(df)} complete samples.")
df.to_csv('deltares_water_2010_2015_200_samples.csv', index=False)

Collected 200 complete samples.


In [6]:
df = pd.read_csv("deltares_water_2010_2015_200_samples.csv")
df.head()

,GrandID,Latitude,Longitude,Sample Date,P,ETa,PET,Melt,Snow,Temp,P_res,S_res,Ea_res,Qin_res,FracFull,Qout_res
0,4132,-24.495832,17.879168,2012-05-10,0.000000,0.405591,0.0,0.000000,0.000000,0.0,0.000000,295000000.0,2.798283,29.839542,1.000000,29.143793
1,4849,24.612499,85.479164,2013-02-04,0.000000,0.097293,0.0,0.000000,0.000000,0.0,0.000000,48515368.0,2.756139,0.061081,0.822294,1.391500
2,5075,30.329166,104.287498,2014-06-27,3.021513,2.289239,0.0,0.000000,0.000000,0.0,4.766330,129911128.0,3.921325,0.074572,0.568042,1.277499
3,592,37.095833,-119.887497,2011-04-12,0.000000,3.015903,0.0,0.221249,3.122778,0.0,0.000000,110875992.0,3.852180,12.269125,0.998883,12.512185
4,1332,22.929167,-98.787498,2010-11-27,0.053035,0.148064,0.0,0.000000,0.000000,0.0,0.066932,30882090.0,2.217035,0.022682,0.156762,0.575446
